Для начала давайте глянем на старую матрицу эмбеддингов модели. Для этого делаем следующее:

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm
from helpers import load_custom_tokenizer
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
DEVICE = 'cuda:0'

model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base')    # Глянем на бейзлайн. 0.6B модель позволяет быстро гонять экспы. В будущем целимся на 4-8B
model.to(DEVICE)

/var/tmp/vasil3vegor/token-distillation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [2]:
old_embs = model.get_input_embeddings().weight
print(old_embs.shape) # изначально в матрице эмебедингов столько токенов. Посмотрим на базовый токенизатор

torch.Size([151936, 1024])


In [3]:
old_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B-Base')
print(len(old_tokenizer))

151669


Заметим, что эти два числа не совпадают. Это нормально, по какой-то причине (толком никому не известной, скорее всего оптимизация какая-то) матрица эмбеддингов часто больше, чем длина словаря токенизатора. Не пугаемся этого.

Давайте посчитаем loss и перплексию модельки на заренее подготовленном наборе данных

In [6]:
from datasets import load_from_disk

eval_ds = load_from_disk('../eval_dataset_fineweb2hq_arb_q07_hash5')
print(eval_ds[0])
print(len(eval_ds))

{'text': 'التناظر\nيسمى كل تطبيق[ر] تقابلي بين مجموعة نقاط مستوٍ (أو نقاط الفضاء)، والمجموعة نفسها تحويلاً نقطياً، ويسمى كل تحويل نقطي يحافظ على الأبعاد تقايساً isometry. والتناظر symmetry (التماثل) هو تقايس يتمتع بخواص معينة.\nقد يكون التقايس مباشراً (محافظاً على التوجيه) كالتناظر بالنسبة لنقطة في مستو أو التناظر بالنسبة لمستقيم في الفضاء، أو غير مباشر (يعكس اتجاه الزوايا) كالتناظر بالنسبة لمستقيم في مستو، أو التناظر بالنسبة لمستو أو نقطة في الفضاء.\nالتناظر في مستو\n1ـ التناظر بالنسبة لمستقيم (ويسمى أيضاً الانعكاس على مستقيم)، ليكن ق مستقيماً معيناً في مستو كـ، ولتكن جميع نقاط هذا المستوي قد رسمت على صحيفة من الورق الشفاف، فإذا قلبت هذه الصحيفة بحيث تبقى كل نقطة من ق في مكانها، فإنه ينتج تقابل بين مجموعة نقاط المستوي وهذه المجموعة نفسها. يسمى هذا التحويل النقطي تناظراً بالنسبة للمستقيم ق أو انعكاساً على المستقيم ق، ويرمز له بـ نق. يدعى المستقيم ق محور التناظر (محور الانعكاس).\nوإذا كان هناك نظام إحداثي م س ع بحيث ينطبق المحور م س على المستقيم ق، ويكون م ع عمودياً عليه. وإذا كانت ن(س،

In [7]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def evaluate_model(model, tokenizer, dataset, max_length=512):

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            padding=False,
        )

    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

    training_args = TrainingArguments(
        output_dir="./eval_tmp",
        per_device_eval_batch_size=8,
        report_to="none",
        dataloader_drop_last=False,
    )

    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        eval_dataset=tokenized,
        data_collator=collator
    )

    metrics = trainer.evaluate()

    loss = metrics["eval_loss"]
    perplexity = torch.exp(torch.tensor(loss)).item()
    metrics["perplexity"] = perplexity

    return loss, perplexity


In [6]:
evaluate_model(model, old_tokenizer, eval_ds)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 2069.63 examples/s]


(2.7459912300109863, 15.580049514770508)

Базовая модель набирает loss 2.74 и перплексию около 15.6

Теперь давайте посмотрим на расширенную модель

In [14]:
new_tokenizer = load_custom_tokenizer('jais_qwen')
print(len(new_tokenizer))
print(f'Added {len(new_tokenizer) - len(old_tokenizer)} new tokens')

199291


NameError: name 'old_tokenizer' is not defined

In [8]:
print(model.get_input_embeddings().weight.shape)
model.resize_token_embeddings(len(new_tokenizer))
print(model.get_input_embeddings().weight.shape)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


torch.Size([151936, 1024])
torch.Size([199291, 1024])


Как мы видим из комментария выше, по дефолту инициализация эмбеддингов берётся при помощи среднего по всем старым эмбеддингам. Давайте оценим эффективность этого способа

In [9]:
loss, perplexity = evaluate_model(model, new_tokenizer, eval_ds)
print(f'Loss: {loss}, perplexity: {perplexity}')

Map: 100%|██████████| 1000/1000 [00:00<00:00, 2943.73 examples/s]


Loss: 13.504070281982422, perplexity: 732391.375


Как мы видим, стало намного хуже)
Чтобы понять насколько, попросим модель что-нибудь сказать нам на арабском. К примеру, продолжить статью из википедии про арабский язык

In [10]:
model.eval()
text = "ٱللُّغَةُ ٱلْعَرَبِيَّة هي أكثر اللغات السامية تحدثًا، وإحدى"

inputs = new_tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print(new_tokenizer.decode(outputs))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

Единственный арабский текст, который тут есть – это сам промт. Дальше модель уходит в полное забытье. Наша задач хоть немного это исправить. Давайте сравним результат инициализации при помощи метода FOCUS, который сейчас используется у нас в пайплайне.

In [15]:
new_embs = torch.load('focus_initialized_embeddings.pt', map_location='cuda:0') # загружаем матрицу эмбеддингов
new_embs = new_embs.to(device=model.device, dtype=model.dtype)

print(new_embs.shape)

torch.Size([199291, 1024])


In [16]:
base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base')
base_model.to('cuda:0')

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

Проверим, что загруженная матрица корректна

In [17]:
def validate_new_embeddings(new_matrix, old_matrix, old_tokenizer_length):
    assert torch.allclose(old_matrix[:old_tokenizer_length], new_matrix[:old_tokenizer_length]) # исходные токены не должны поменяться
    assert new_embs.shape[0] == len(new_tokenizer)

In [21]:
old_tokenizer_length = len(old_tokenizer)
old_matrix = base_model.get_input_embeddings().weight

validate_new_embeddings(new_embs, old_matrix, old_tokenizer_length)

Подменяем матрицу эмбеддингов нашей модели:

In [22]:
model.get_input_embeddings().weight.data = new_embs

In [25]:
model.resize_token_embeddings(new_embs.shape[0])

Embedding(151936, 1024)

In [26]:
model.get_input_embeddings().weight.data.copy_(new_embs)

tensor([[-3.0975e-03,  3.2715e-02, -7.0312e-02,  ...,  1.3794e-02,
         -1.4404e-02,  1.2817e-02],
        [ 3.0273e-02,  2.4414e-02, -6.1279e-02,  ..., -3.1128e-03,
         -3.7354e-02,  7.7209e-03],
        [ 2.9175e-02,  3.2227e-02, -2.2339e-02,  ..., -9.5215e-03,
          2.5482e-03,  2.5635e-02],
        ...,
        [ 1.6724e-02, -3.4180e-02,  5.2002e-02,  ...,  2.8076e-02,
         -5.4321e-03, -1.9653e-02],
        [ 8.4839e-03, -5.8594e-02,  3.5400e-02,  ..., -6.2943e-05,
          1.2024e-02, -5.9509e-03],
        [ 3.3264e-03, -5.6152e-02,  1.6602e-02,  ...,  1.6968e-02,
          3.3447e-02, -1.6235e-02]], device='cuda:0', grad_fn=<CopyBackwards>)

In [27]:
if model.config.tie_word_embeddings:
    model.tie_weights()

In [28]:
evaluate_model(model, new_tokenizer, eval_ds)

(6.419434070587158, 613.6557006835938)

Как мы видим, loss действительно уменьшился на порядок. Однако без дообучения заставить модель говорить что-то внятное всё равно почти невозможно)
Посмотрим, что она генерит в таком состоянии:

In [17]:
model.eval()
text = "ٱللُّغَةُ ٱلْعَرَبِيَّة هي أكثر اللغات السامية تحدثًا، وإحدى"

inputs = new_tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print(new_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 تحدثت في أوروبا, وتركيا, وجنوب إفريقيا, وجنوب إفريقيا, وجنوب إفريقيا, وجنوب إفريقيا, وجن


Перевод:
*Я выступал в Европе, Турции, Южной Африке, Южной Африке, Южной Африке и Южной Африке.*

Как мы видим, модель не понимает, что она говорит, но понимает, что должна говорить по-арабски. И это хорошо

In [5]:
def res(path: str):
    model = AutoModelForCausalLM.from_pretrained(path).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(path)

    loss, ppl = evaluate_model(model, tokenizer, eval_ds)
    print("metrics:", loss, ppl)
